# Memory considerations when training neural networks on GPUs

Use ResNet-152 for parameter and optimizer-memory measurements (Sections 1-2),
then ResNet-18 for the batch-size experiment (Section 3). Answer the questions in
your written submission. There is no gradient-accumulation exercise in this notebook.

**Runtime:** select a Colab GPU runtime. A CPU can check the code but cannot answer
the GPU-memory questions. Record your GPU, PyTorch version, dtype, and batch sizes.
All numbers labeled **MB** use 1 MB = 1,000,000 bytes; `nvidia-smi` displays MiB,
so convert MiB to MB before comparing with these measurements. The code queries
CUDA memory in bytes and converts those values to MB. Results depend on hardware and software.

Run cells in order. Section 2 runs only two minibatches per optimizer. Section 3
trains for 10 epochs per batch size and can take substantially longer. Logs stay
under `hw06_memory/logs` in the current session; download them before Colab resets.


In [ ]:
# Colab already supplies a compatible torch/torchvision pair. Do not upgrade one alone.
%pip -q install pandas matplotlib


In [ ]:
import gc
import os
import random
import subprocess
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms

SMOKE_TEST = os.environ.get('HW06_SMOKE_TEST', '0') == '1'
DATA_ROOT = Path(os.environ.get('HW06_DATA_ROOT', './data'))
ROOT_PATH = Path(os.environ.get('HW06_OUTPUT_ROOT', './hw06_memory'))
LOG_DIR = ROOT_PATH / ('smoke_logs' if SMOKE_TEST else 'logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEED = 42
PROFILE_BATCH_SIZE = 2 if SMOKE_TEST else 8
EPOCHS = 1 if SMOKE_TEST else 10
MAX_BATCHES = 2 if SMOKE_TEST else None

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

def cleanup_memory():
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()  # Releases unused cached blocks, not live tensors.

def memory_snapshot():
    if device.type != 'cuda':
        return dict(allocated_mb=np.nan, reserved_mb=np.nan, peak_allocated_mb=np.nan,
                    device_used_mb=np.nan)
    synchronize()
    result = dict(allocated_mb=torch.cuda.memory_allocated(device) / 1e6,
                  reserved_mb=torch.cuda.memory_reserved(device) / 1e6,
                  peak_allocated_mb=torch.cuda.max_memory_allocated(device) / 1e6,
                  device_used_mb=np.nan)
    # Query the selected CUDA device, respecting CUDA_VISIBLE_DEVICES.
    # Device-wide usage also includes the driver and other processes.
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    result['device_used_mb'] = (total_bytes - free_bytes) / 1e6
    return result

seed_everything()
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(device), 'total MB:',
          torch.cuda.get_device_properties(device).total_memory / 1e6)
else:
    print('CPU mode: GPU memory measurements will be unavailable (NaN).')


## Data

We resize CIFAR-10 images to 224 x 224 **to make memory costs visible**. ResNets
with adaptive average pooling can accept other spatial sizes; 224 x 224 is not a
hard input-shape requirement. We select the same subset without replacement for
all runs. Models start from the same random seed. No pretrained weights are used.


In [ ]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
data_train = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=True,
                                         download=True, transform=transform)
subset_size = 32 if SMOKE_TEST else 4096
indices = torch.randperm(len(data_train), generator=torch.Generator().manual_seed(SEED))[:subset_size]
subsampled_train_data = torch.utils.data.Subset(data_train, indices.tolist())
print('Training subset:', len(subsampled_train_data))
print('SMOKE_TEST:', SMOKE_TEST, '| epochs:', EPOCHS, '| max batches:', MAX_BATCHES,
      '| dtype: float32 | logs:', LOG_DIR.resolve())


## 1. Parameters, CUDA runtime overhead, and memory accounting

Training memory includes parameters, saved activations, parameter gradients,
optimizer state, input tensors, and temporary workspaces. At fixed architecture
and dtype, **parameter-gradient tensors have the same shape as the parameters**:
their storage does not grow with batch size. Activations and their intermediate
gradients generally do grow with batch size.

PyTorch's **allocated** memory counts live tensors; **reserved** memory includes
unused blocks held by its caching allocator. `nvidia-smi` reports device-wide
usage, including driver/library allocations and other processes. Thus subtracting
parameter bytes from device-wide usage does **not** isolate the CUDA context.


In [ ]:
cleanup_memory()
before_model = memory_snapshot()
model = torchvision.models.resnet152(weights=None, num_classes=10).to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
parameter_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
buffer_mb = sum(b.numel() * b.element_size() for b in model.buffers()) / 1e6
after_model = memory_snapshot()
print('Trainable parameters:', parameter_count)
print('Parameter storage (MB):', parameter_mb, '| buffer storage (MB):', buffer_mb)
print(pd.DataFrame([before_model, after_model], index=['before model', 'after model']))
print('Device-wide usage minus parameter storage (MB):',
      after_model['device_used_mb'] - parameter_mb)
# Remove this reference before optimizer experiments, otherwise two models remain live.
del model
cleanup_memory()


### Written questions

1a. How many trainable parameters does this **10-class** ResNet-152 have? Estimate
its float32 parameter storage in MB (excluding buffers, gradients, and optimizer state).

1b. Which GPU are you using and how much total memory does it have?

1c. Report device-wide GPU usage minus parameter storage after loading the model.
Why is this only a rough overhead estimate, rather than an isolated CUDA-context measurement?
If the driver does not expose device-wide usage, report that limitation and the
allocated/reserved measurements instead.


## 2. Optimizer memory

Plain SGD has no persistent per-parameter state beyond parameters/gradients.
Momentum SGD stores one momentum tensor per parameter; Adam normally stores two
moment tensors plus small step counters. State is allocated lazily at the first
step. Temporary optimizer buffers and library workspaces can affect peak memory.

We use the same model, float32 dtype, batch size, and two batches for every optimizer.
The second step helps capture steady-state optimizer behavior. Each experiment
deletes its model and optimizer before the next one. Unused cached blocks are
released, but CUDA/library state can persist; use a fresh runtime for precise
device-wide comparisons and record background GPU activity.


In [ ]:
def make_optimizer(name, parameters, lr=0.001):
    if name == 'SGD':
        return torch.optim.SGD(parameters, lr=lr)
    if name == 'SGD_WITH_MOMENTUM':
        return torch.optim.SGD(parameters, lr=lr, momentum=0.9)
    if name == 'ADAM':
        return torch.optim.Adam(parameters, lr=lr)
    raise ValueError(f'Unknown optimizer: {name}')

def profile_mem_usage(optimizer_name, batch_size=PROFILE_BATCH_SIZE):
    cleanup_memory()
    seed_everything()
    model = optimizer = images = labels = outputs = loss = None
    try:
        model = torchvision.models.resnet152(weights=None, num_classes=10).to(device)
        optimizer = make_optimizer(optimizer_name, model.parameters())
        loader = torch.utils.data.DataLoader(subsampled_train_data, batch_size=batch_size,
                                             shuffle=False, num_workers=0)
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        model.train()
        for step, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = torch.nn.functional.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            if step == 1:
                break
        state_mb = sum(v.numel() * v.element_size() for state in optimizer.state.values()
                       for v in state.values() if torch.is_tensor(v)) / 1e6
        row = dict(optimizer=optimizer_name, batch_size=batch_size, status='ok',
                   smoke_test=SMOKE_TEST, dtype='float32', steps=step + 1,
                   optimizer_state_mb=state_mb, **memory_snapshot())
        pd.DataFrame([row]).to_csv(LOG_DIR / f'resnet152__{optimizer_name}.csv', index=False)
        return row
    except torch.cuda.OutOfMemoryError:
        row = dict(optimizer=optimizer_name, batch_size=batch_size, status='out of memory',
                   smoke_test=SMOKE_TEST, dtype='float32')
        pd.DataFrame([row]).to_csv(LOG_DIR / f'resnet152__{optimizer_name}.csv', index=False)
        return row
    finally:
        del model, optimizer, images, labels, outputs, loss
        cleanup_memory()

optimizer_results = pd.DataFrame([profile_mem_usage(name) for name in
                                 ['SGD', 'SGD_WITH_MOMENTUM', 'ADAM']])
print(optimizer_results.to_string(index=False))
if not (optimizer_results['status'] == 'ok').all():
    print('Reduce PROFILE_BATCH_SIZE and rerun ALL three experiments at the same batch size.')


### Written questions

2a. Report peak PyTorch allocated memory, post-step reserved memory, and device-wide
usage (when available), in MB, for all three optimizers. Use the same batch size.

2b. Which optimizer has the largest persistent state? Explain the difference
between state size and the measured total/peak memory.


## 3. Batch size, accuracy, and wall-clock time

Use **ResNet-18** here. Batch size affects input/activation storage and the number
of optimizer steps per epoch. Larger batches can improve GPU throughput, but the
fastest batch size depends on hardware, model, data loading, and memory limits;
it is not necessarily the largest that fits.

The learning-rate table uses linear scaling through batch size 256 and then caps
the rate. This is an empirical heuristic, not a guarantee of stability or higher
accuracy. Because both batch size and learning rate change, do not attribute all
accuracy differences to batch size alone. A controlled extension would keep the
learning rate fixed or tune it separately for each batch size.

We record **training accuracy**, not test accuracy, to reproduce the requested
plot. Timing starts before epoch 1 and includes training, the training-set
evaluation after each epoch, and measurement/logging overhead. It is end-to-end
experiment time, not isolated GPU kernel time. CUDA is synchronized at timing boundaries.


In [ ]:
LR_MAP = {4: 0.0001, 8: 0.0002, 16: 0.0004, 32: 0.0008,
          64: 0.0016, 128: 0.0032, 256: 0.0064, 512: 0.0064, 1024: 0.0064}

@torch.no_grad()
def accuracy_percent(model, loader, max_batches=None):
    model.eval()
    correct = count = 0
    for step, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        count += len(labels)
        if max_batches is not None and step + 1 >= max_batches:
            break
    return 100 * correct / count

def run_train(batch_size, epochs=EPOCHS, max_batches=MAX_BATCHES):
    cleanup_memory()
    seed_everything()
    model = optimizer = images = labels = outputs = loss = None
    rows = []
    path = LOG_DIR / f'resnet18__{batch_size}.csv'
    try:
        model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=LR_MAP[batch_size], momentum=0.9)
        loader = torch.utils.data.DataLoader(subsampled_train_data, batch_size=batch_size,
            shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(SEED))
        eval_loader = torch.utils.data.DataLoader(subsampled_train_data, batch_size=batch_size,
                                                  shuffle=False, num_workers=0)
        synchronize()
        start = time.perf_counter()
        for epoch in range(epochs):
            if device.type == 'cuda':
                torch.cuda.reset_peak_memory_stats(device)
            model.train()
            total_loss = count = 0
            for step, (images, labels) in enumerate(loader):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                outputs = model(images)
                loss = torch.nn.functional.cross_entropy(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(labels)
                count += len(labels)
                if max_batches is not None and step + 1 >= max_batches:
                    break
            # Capture training memory before evaluation, then free the last graph/tensors.
            stats = memory_snapshot()
            images = labels = outputs = loss = None
            accuracy = accuracy_percent(model, eval_loader, max_batches)
            synchronize()
            row = dict(epoch=epoch + 1, batch_size=batch_size, learning_rate=LR_MAP[batch_size],
                       walltime=time.perf_counter() - start, loss=total_loss / count,
                       accuracy=accuracy, status='ok', smoke_test=SMOKE_TEST,
                       subset_size=len(subsampled_train_data), requested_epochs=epochs,
                       max_batches=max_batches, dtype='float32', **stats)
            rows.append(row)
            pd.DataFrame(rows).to_csv(path, index=False)
            print(f'Batch {batch_size}, epoch {epoch+1}: loss={row["loss"]:.4f}, '
                  f'train accuracy={accuracy:.2f}%, time={row["walltime"]:.1f}s')
        return pd.DataFrame(rows)
    except torch.cuda.OutOfMemoryError:
        print(f'Batch size {batch_size}: out of memory; record this outcome.')
        rows.append(dict(batch_size=batch_size, epoch=len(rows), status='out of memory',
                         smoke_test=SMOKE_TEST, subset_size=len(subsampled_train_data),
                         requested_epochs=epochs, max_batches=max_batches, dtype='float32'))
        frame = pd.DataFrame(rows)
        frame.to_csv(path, index=False)
        return frame
    finally:
        del model, optimizer, images, labels, outputs, loss
        cleanup_memory()


In [ ]:
batch_sizes = [4, 16] if SMOKE_TEST else [4, 16, 64, 256]
# Optionally try 512 or 1024 AFTER completing the required sizes that fit.
batch_results = {batch_size: run_train(batch_size) for batch_size in batch_sizes}


In [ ]:
plt.figure(figsize=(7, 4))
for batch_size, frame in batch_results.items():
    if 'walltime' in frame:
        plt.plot(frame['walltime'], frame['accuracy'], marker='o', label=f'batch={batch_size}')
plt.xlabel('End-to-end elapsed experiment time (s), from start of epoch 1')
plt.ylabel('Training accuracy (%)')
plt.title('Training accuracy vs wall-clock time')
if any('walltime' in frame for frame in batch_results.values()):
    plt.legend()
plt.tight_layout()
plt.show()
summary = pd.concat([frame.tail(1) for frame in batch_results.values()], ignore_index=True)
print(summary.to_string(index=False))
print('Logs:', LOG_DIR.resolve())


### Written questions

3a. Report the same memory metrics for batch sizes 4, 16, 64, and 256. Which sizes
fit? What was the largest batch size you actually tested successfully? An OOM
result is valid; there is no universal threshold.

3b. Which batch size achieved the highest **training accuracy** after 10 epochs?
Report the associated learning rate. Do not infer test accuracy from this result.

3c. Which completed 10 epochs fastest? Explain using throughput, number of updates,
and hardware/data-loading limits. Include actual elapsed time.

3d. Include the training-accuracy-versus-wall-time plot in your written submission.
Smoke-test results are for checking the code and are not the homework experiment.

Reference: [PyTorch CUDA memory management](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management).
